# State Objective Rebound Target Enrichment v1

Goal: continue from `state_objective_rebound_weighted_bc_v1`, where deterministic rebound loss weighting did not improve the sparse rebound slice. This notebook tests whether enriching the rebound target labels themselves fixes the failure before any PPO training.

External basis:
- Qlib: repeatable quantitative investment workflow and artifact discipline: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL: keep train / validate / test separation before any DRL promotion: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Demonstration / BC pretraining remains a warm-start concept only: https://arxiv.org/abs/1704.03732
- Class imbalance literature motivates checking sparse-label remedies, but the screen must reject remedies that only lift global/downtrend metrics: https://arxiv.org/abs/1708.02002 and https://arxiv.org/abs/1901.05555

Governance:
- This notebook is a no-promotion target-screen. It does not backtest and does not create a runtime actor.
- Enrichment only modifies rows that already exist in the fold-native target table; it does not invent future dates or cross fold/window boundaries.
- If enrichment cannot lift original rebound accuracy, the next valid step is state-conditioned objective/model structure, not wider enrichment grids.


In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SPEC.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor,
    fit_bc_actor,
)
from src.soxl_quant.app.state_objective_training_targets import (
    StateObjectiveReboundTargetEnrichmentConfig,
    enrich_state_objective_rebound_targets,
)
from src.soxl_quant.config import load_profile
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import (
    build_policy,
    build_state_objective_warm_start_dataset_from_frame,
)

EXPERIMENT = "state_objective_rebound_target_enrichment_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
SOURCE_CARD = ROOT / "runtime" / "model_selection" / "card" / "state_objective_rebound_weighted_bc_v1"
TEACHER_CARD = ROOT / "runtime" / "model_selection" / "card" / "state_objective_fold_native_teacher_router_v1"
TARGETS_PATH = TEACHER_CARD / "fold_native_teacher_targets.csv"
FRAME_PATH = ROOT / "data" / "soxl_profile_balanced_rule_residual_v22.pkl"
PROFILE_NAME = "risk_budget_usage_v1"
CARD.mkdir(parents=True, exist_ok=True)

source_selection = json.loads((SOURCE_CARD / "selection.json").read_text())
profile = load_profile(PROFILE_NAME)
frame = pd.read_pickle(FRAME_PATH)
targets = pd.read_csv(TARGETS_PATH, parse_dates=["datetime"])
original_rebound_targets = targets[targets["is_rebound_cluster"].astype(bool)]
original_downtrend_targets = targets[targets["is_downtrend_state"].astype(bool)]
original_ordinary_targets = targets[
    (~targets["is_rebound_cluster"].astype(bool))
    & (~targets["is_downtrend_state"].astype(bool))
]

base_dataset = build_state_objective_bc_dataset_from_frame(
    frame,
    targets=targets,
    profile=profile,
    sample_size=0,
)
base_summary = {
    "profile": PROFILE_NAME,
    "feature_frame": str(FRAME_PATH.relative_to(ROOT)),
    "source_experiment": "state_objective_rebound_weighted_bc_v1",
    "source_decision": source_selection.get("decision"),
    "base_samples": int(len(base_dataset.actions)),
    "base_selected_rebound_rows": int(base_dataset.metadata["is_rebound_cluster"].sum()),
    "base_target_rebound_rows": int(original_rebound_targets.shape[0]),
    "base_selected_downtrend_rows": int(base_dataset.metadata["is_downtrend_state"].sum()),
}
(CARD / "base_summary.json").write_text(json.dumps(base_summary, indent=2, sort_keys=True), encoding="utf-8")
base_summary

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'profile': 'risk_budget_usage_v1',
 'feature_frame': 'data/soxl_profile_balanced_rule_residual_v22.pkl',
 'source_experiment': 'state_objective_rebound_weighted_bc_v1',
 'source_decision': 'rebound_weighting_no_improvement_no_promotion',
 'base_samples': 755,
 'base_selected_rebound_rows': 4,
 'base_target_rebound_rows': 7,
 'base_selected_downtrend_rows': 477}

## Enrichment Screen

The screen varies only two target-enrichment knobs: neighbor radius and rebound floor. Each run reinitializes the same `risk_budget_usage_v1` actor with the same seed. The key metric is original rebound accuracy; enriched rebound accuracy is diagnostic only because it includes synthetic neighbor labels.


In [2]:
def evaluate_target_subset(actor, label: str, subset: pd.DataFrame) -> dict[str, object]:
    if subset.empty:
        return {"slice": label, "samples": 0, "loss": np.nan, "accuracy": np.nan}
    obs, actions = build_state_objective_warm_start_dataset_from_frame(
        frame,
        targets=subset,
        profile=profile,
        sample_size=0,
    )
    metrics = evaluate_bc_actor(actor, obs, actions)
    metrics["slice"] = label
    return metrics

screen_rows = []
state_rows = []
radius_grid = [0, 1, 2, 3, 5, 8]
floor_grid = [0.35, 0.55, 0.75]
train_config = StateObjectiveBCTrainingConfig(steps=600, lr=0.001, batch_size=128, seed=42)

for radius in radius_grid:
    for min_floor in floor_grid:
        enriched = enrich_state_objective_rebound_targets(
            targets,
            StateObjectiveReboundTargetEnrichmentConfig(
                radius=int(radius),
                min_target_position=float(min_floor),
                objective_weight_decay=0.50,
                exclude_downtrend=True,
            ),
        )
        dataset = build_state_objective_bc_dataset_from_frame(
            frame,
            targets=enriched,
            profile=profile,
            sample_size=0,
        )
        torch.manual_seed(int(train_config.seed))
        _policy, actor = build_policy(observation_dim_for_profile(profile), profile)
        fit_metrics = fit_bc_actor(actor, dataset.obs, dataset.actions, train_config)
        enriched_rebound_targets = enriched[enriched["is_rebound_cluster"].astype(bool)]
        enriched_ordinary_targets = enriched[
            (~enriched["is_rebound_cluster"].astype(bool))
            & (~enriched["is_downtrend_state"].astype(bool))
        ]
        row = {
            "radius": int(radius),
            "min_target_position": float(min_floor),
            "target_rebound_rows": int(enriched["is_rebound_cluster"].astype(bool).sum()),
            "selected_rebound_rows": int(dataset.metadata["is_rebound_cluster"].sum()),
            "new_enriched_rows": int((enriched["rebound_enrichment_source"] == "neighbor_rebound").sum()),
            "unique_actions": int(len(set(dataset.actions.tolist()))),
            **fit_metrics,
        }
        subsets = {
            "original_rebound": original_rebound_targets,
            "enriched_rebound": enriched_rebound_targets,
            "original_downtrend": original_downtrend_targets,
            "original_ordinary": original_ordinary_targets,
            "enriched_ordinary": enriched_ordinary_targets,
        }
        for name, subset in subsets.items():
            metrics = evaluate_target_subset(actor, name, subset)
            row[f"{name}_accuracy"] = float(metrics["accuracy"])
            row[f"{name}_loss"] = float(metrics["loss"])
            row[f"{name}_samples"] = int(metrics["samples"])
            state_rows.append({
                "radius": int(radius),
                "min_target_position": float(min_floor),
                **metrics,
            })
        screen_rows.append(row)

screen = pd.DataFrame(screen_rows)
state_metrics = pd.DataFrame(state_rows)
screen["original_rebound_gap"] = (0.60 - screen["original_rebound_accuracy"]).clip(lower=0.0)
screen["global_gap"] = (0.80 - screen["after_accuracy"]).clip(lower=0.0)
screen["ordinary_gap"] = (0.65 - screen["original_ordinary_accuracy"]).clip(lower=0.0)
screen["enrichment_loss"] = (
    10.0 * screen["original_rebound_gap"]
    + 2.0 * screen["global_gap"]
    + 2.0 * screen["ordinary_gap"]
    - 0.05 * screen["original_downtrend_accuracy"]
)
screen = screen.sort_values(
    ["enrichment_loss", "original_rebound_gap", "after_accuracy"],
    ascending=[True, True, False],
).reset_index(drop=True)
screen.to_csv(CARD / "enrichment_screen.csv", index=False)
state_metrics.to_csv(CARD / "state_slice_metrics.csv", index=False)
screen

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


,radius,min_target_position,target_rebound_rows,selected_rebound_rows,new_enriched_rows,unique_actions,samples,steps,lr,batch_size,...,original_ordinary_accuracy,original_ordinary_loss,original_ordinary_samples,enriched_ordinary_accuracy,enriched_ordinary_loss,enriched_ordinary_samples,original_rebound_gap,global_gap,ordinary_gap,enrichment_loss
0,8,0.55,19,9,18,5,755,600,0.001,128,...,0.737410,1.173067,278,0.737410,1.173067,278,0.171429,0.0,0.0,1.673510
1,8,0.75,19,9,18,5,755,600,0.001,128,...,0.737410,1.173067,278,0.737410,1.173067,278,0.171429,0.0,0.0,1.673510
2,0,0.35,7,4,0,5,755,600,0.001,128,...,0.723022,1.174270,278,0.723022,1.174270,278,0.171429,0.0,0.0,1.673615
3,0,0.55,7,4,0,5,755,600,0.001,128,...,0.723022,1.174270,278,0.723022,1.174270,278,0.171429,0.0,0.0,1.673615
4,0,0.75,7,4,0,5,755,600,0.001,128,...,0.723022,1.174270,278,0.723022,1.174270,278,0.171429,0.0,0.0,1.673615
5,1,0.35,10,5,9,5,755,600,0.001,128,...,0.723022,1.174270,278,0.723022,1.174270,278,0.171429,0.0,0.0,1.673615
6,1,0.55,10,5,9,5,755,600,0.001,128,...,0.723022,1.174270,278,0.723022,1.174270,278,0.171429,0.0,0.0,1.673615
7,1,0.75,10,5,9,5,755,600,0.001,128,...,0.723022,1.174270,278,0.723022,1.174270,278,0.171429,0.0,0.0,1.673615
8,2,0.35,12,6,11,5,755,600,0.001,128,...,0.723022,1.174270,278,0.723022,1.174270,278,0.171429,0.0,0.0,1.673615
9,2,0.55,12,6,11,5,755,600,0.001,128,...,0.723022,1.174270,278,0.723022,1.174270,278,0.171429,0.0,0.0,1.673615


In [3]:
screen.pivot_table(
    index="radius",
    columns="min_target_position",
    values="original_rebound_accuracy",
    aggfunc="first",
).reset_index()

min_target_position,radius,0.35,0.55,0.75
0,0,0.428571,0.428571,0.428571
1,1,0.428571,0.428571,0.428571
2,2,0.428571,0.428571,0.428571
3,3,0.428571,0.428571,0.428571
4,5,0.428571,0.428571,0.428571
5,8,0.428571,0.428571,0.428571


## Decision

A pass requires improvement on the original rebound rows, not only better fit to enriched synthetic neighbors. If the original rebound slice stays flat, enrichment is not a valid handoff into PPO.


In [4]:
best = screen.iloc[0].to_dict()
source_rebound_accuracy = float(source_selection.get("best_rebound_accuracy", source_selection.get("source_rebound_accuracy", np.nan)))
rebound_improvement = float(best["original_rebound_accuracy"] - source_rebound_accuracy)
full_gate = bool(
    best["original_rebound_accuracy"] >= 0.60
    and best["after_accuracy"] >= 0.80
    and best["original_ordinary_accuracy"] >= 0.65
)
partial_improvement = bool(rebound_improvement > 0.0 and best["after_accuracy"] >= 0.80)
if full_gate:
    decision = "rebound_target_enrichment_passed_diagnostic_gate"
elif partial_improvement:
    decision = "rebound_target_enrichment_partial_improvement_no_promotion"
else:
    decision = "rebound_target_enrichment_no_improvement_no_promotion"

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "profile": PROFILE_NAME,
    "source_experiment": "state_objective_rebound_weighted_bc_v1",
    "promotion_allowed": False,
    "promotion_blocker": "Target enrichment is not a backtest and did not improve original rebound fit; no PPO handoff promotion from this screen.",
    "full_gate": full_gate,
    "partial_improvement": partial_improvement,
    "source_rebound_accuracy": source_rebound_accuracy,
    "best_original_rebound_accuracy": float(best["original_rebound_accuracy"]),
    "best_enriched_rebound_accuracy": float(best["enriched_rebound_accuracy"]),
    "rebound_improvement": rebound_improvement,
    "best_radius": int(best["radius"]),
    "best_min_target_position": float(best["min_target_position"]),
    "best_all_accuracy": float(best["after_accuracy"]),
    "best_downtrend_accuracy": float(best["original_downtrend_accuracy"]),
    "best_ordinary_accuracy": float(best["original_ordinary_accuracy"]),
    "best_target_rebound_rows": int(best["target_rebound_rows"]),
    "best_selected_rebound_rows": int(best["selected_rebound_rows"]),
    "next_mechanism": "state_conditioned_loss_or_model_structure_before_walkforward_ppo",
    "acceptance_gate_for_next_run": {
        "relative_return_min_pp": 30.0,
        "max_dd_max_pct": 15.0,
        "walkforward_required": True,
        "fixed_split_required": True,
        "original_rebound_accuracy_min": 0.60,
        "avoid_more_neighbor_enrichment_grids": True,
    },
    "sidecars": [
        "base_summary.json",
        "enrichment_screen.csv",
        "state_slice_metrics.csv",
        "summary.json",
    ],
    "external_sources": [
        "https://github.com/microsoft/qlib",
        "https://arxiv.org/abs/2009.11189",
        "https://github.com/AI4Finance-Foundation/FinRL",
        "https://arxiv.org/abs/2011.09607",
        "https://arxiv.org/abs/1704.03732",
        "https://arxiv.org/abs/1708.02002",
        "https://arxiv.org/abs/1901.05555",
    ],
}
summary = {
    "conclusion": [
        f"Best enrichment config radius={best['radius']}, floor={best['min_target_position']:.2f}; original rebound accuracy {best['original_rebound_accuracy']:.3f} vs source {source_rebound_accuracy:.3f}, improvement {rebound_improvement:.3f}.",
        f"Best all/downtrend/original ordinary accuracies were {best['after_accuracy']:.3f}/{best['original_downtrend_accuracy']:.3f}/{best['original_ordinary_accuracy']:.3f}.",
        f"Target rebound rows expanded to {int(best['target_rebound_rows'])}, but original rebound fit did not improve; neighbor enrichment is not a valid PPO-ready warm start.",
        "Next step should move to state-conditioned loss/model structure or richer rebound target discovery, not wider neighbor/floor grids.",
    ],
    "base_summary": base_summary,
    "best_row": best,
    "selection": selection,
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True), encoding="utf-8")
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
selection

{'experiment': 'state_objective_rebound_target_enrichment_v1',
 'created_at': '2026-05-25T12:43:18.978048+00:00',
 'decision': 'rebound_target_enrichment_no_improvement_no_promotion',
 'profile': 'risk_budget_usage_v1',
 'source_experiment': 'state_objective_rebound_weighted_bc_v1',
 'promotion_allowed': False,
 'promotion_blocker': 'Target enrichment is not a backtest and did not improve original rebound fit; no PPO handoff promotion from this screen.',
 'full_gate': False,
 'partial_improvement': False,
 'source_rebound_accuracy': 0.42857142857142855,
 'best_original_rebound_accuracy': 0.42857142857142855,
 'best_enriched_rebound_accuracy': 0.2631578947368421,
 'rebound_improvement': 0.0,
 'best_radius': 8,
 'best_min_target_position': 0.55,
 'best_all_accuracy': 0.8370860927152318,
 'best_downtrend_accuracy': 0.8155136268343816,
 'best_ordinary_accuracy': 0.737410071942446,
 'best_target_rebound_rows': 19,
 'best_selected_rebound_rows': 9,
 'next_mechanism': 'state_conditioned_loss_